# Optimizers: SGD, SGD+Momentum, Adam, AdamW

We compare convergence on the fraud dataset using a small MLP. Goal: build intuition for **which optimizer to reach for** and what their failure modes look like.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet').sample(10_000, random_state=SEED)
df = pd.get_dummies(df, columns=['device_type', 'country'], drop_first=True)
for c in ['email_risk', 'device_entropy']:
    df[c] = df[c].fillna(df[c].median())

y = df['is_fraud'].values.astype(np.float32)
X = df.drop(columns=['is_fraud']).values.astype(np.float32)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_tr)
X_tr, X_va = scaler.transform(X_tr).astype(np.float32), scaler.transform(X_va).astype(np.float32)
print(X_tr.shape, X_va.shape)

In [ ]:
def make_loader(X, y, batch_size=128):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y).unsqueeze(1))
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

def make_model(in_dim):
    return nn.Sequential(
        nn.Linear(in_dim, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    )

def train_one(opt_name, epochs=8):
    torch.manual_seed(SEED)
    model = make_model(X_tr.shape[1])
    if opt_name == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=0.05)
    elif opt_name == 'sgd_momentum':
        opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    elif opt_name == 'adam':
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    elif opt_name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()
    tr_loader = make_loader(X_tr, y_tr)
    val_losses = []
    for ep in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(torch.from_numpy(X_va)), torch.from_numpy(y_va).unsqueeze(1)).item()
        val_losses.append(v)
    return val_losses

curves = {name: train_one(name) for name in ['sgd', 'sgd_momentum', 'adam', 'adamw']}
for name, vals in curves.items():
    print(f"{name:14s}  final val loss: {vals[-1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 4))
for name, vals in curves.items():
    plt.plot(range(1, len(vals)+1), vals, marker='o', label=name)
plt.xlabel('Epoch'); plt.ylabel('Val BCE loss')
plt.title('Optimizer comparison on the fraud MLP')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## What you should observe

- **Plain SGD** converges slowest — single learning rate for all params, no momentum to push through flat regions.
- **SGD + momentum** accelerates and typically matches or beats Adam at convergence (with proper LR tuning).
- **Adam / AdamW** start fast — adaptive per-parameter LR shines early.
- **AdamW** vs Adam: weight decay in AdamW is applied directly to weights, not added to the gradient. This matters a lot for transformers and well-regularized vision nets; on small MLPs the difference is modest.

**Interview soundbite:** *'AdamW for any modern deep net — its decoupled weight decay regularizes more predictably than Adam's L2-in-gradient. SGD+momentum still wins for vision when you can afford to tune carefully.'*